# notebooks/05a_gnn_architecture.ipynb — Markdown Cell

# Phase 5A — Deterministic GNN Architecture

## Phase Status

This notebook analyzes deterministic GNN models trained in Phase 5A.

Phase 5A is the first part of the main research contribution. It evaluates whether graph-based message passing improves wildfire burn probability prediction compared with the strongest non-graph baselines from Phase 4 and Phase 4B.

---

## 1. Scientific Motivation

Previous phases established strong baselines:

- Random Forest and XGBoost use tabular node features.
- CNN uses local raster patches and convolutional spatial context.
- GNN uses explicit graph topology and message passing.

The key scientific question in Phase 5A is:

> Does graph topology add predictive value beyond tabular features and CNN spatial convolution?

This is important because the strongest baseline is now the CNN spatial model.

```text
Best Phase 4 model: XGBoost, R² = 0.6761
Best Phase 4B model: CNN, R² = 0.7187

GCN

>python scripts/phase5a_run_gnn.py --model gcn --epochs 50 --device cpu


===========================================================================
Phase 5A — Deterministic GNN Architecture
===========================================================================
Loading graph: D:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Nodes     : 327,405
Features  : 61
Edges     : 2,511,084
Train     : 237,304
Val       : 32,570
Test      : 57,531

Model architecture:
GCNModel(
  (convs): ModuleList(
    (0): GCNConv(61, 64)
    (1-2): 2 x GCNConv(64, 64)
  )
  (head): MLPHead(
    (net): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=64, out_features=1, bias=True)
    )
  )
)

Training started...
Epoch 001/50 | train_loss=5614.34375 | val_loss=13449.08105 | lr=1.00e-03
Epoch 005/50 | train_loss=527.05066 | val_loss=903.23206 | lr=1.00e-03
Epoch 010/50 | train_loss=155.96542 | val_loss=3.45268 | lr=1.00e-03
Epoch 015/50 | train_loss=46.73463 | val_loss=3.61016 | lr=1.00e-03
Epoch 020/50 | train_loss=13.01342 | val_loss=0.79873 | lr=1.00e-03
Epoch 025/50 | train_loss=3.91892 | val_loss=1.55369 | lr=1.00e-03
Epoch 030/50 | train_loss=1.51573 | val_loss=1.36348 | lr=5.00e-04
Early stopping at epoch 34

Training finished in 227.8s
Best val_loss: 0.73921

Predicting on test split...

  ── GCN ──
    R²       =  -0.0179   (>0 = beats naive mean predictor)
    MAE      =  0.02182  (burn prob scale ~[0, 0.25])
    Spearman =   0.6773   (rank correlation, robust)
    Brier    =  0.00142  (lower = better calibration)
    ECE      =  0.00936  (target < 0.05 after calibration)
    n_test   =   57,531

    Binned by BP quantile:
    Bin    BP range                      n       R²        MAE   Spearman
    --------------------------------------------------------------------
    1      [0.0000, 0.0040]    11,506 -152.023    0.00917      0.083
    2      [0.0040, 0.0099]    11,505 -122.540    0.01018      0.282
    3      [0.0099, 0.0205]    11,507 -107.139    0.01538      0.239
    4      [0.0205, 0.0465]    11,506  -26.181    0.01975      0.043
    5      [0.0465, 0.2082]    11,507   -2.007    0.05463      0.390 ← HIGH RISK

===========================================================================
Phase 5A complete
===========================================================================
Saved metrics     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gcn_metrics.csv
Saved binned      : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gcn_binned_metrics.csv
Saved history     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gcn_history.csv
Saved predictions : D:\wildfire\spatiotemporal_wildfire_gnn\reports\predictions\phase5a_gcn_test_predictions.npz
Saved checkpoint  : D:\wildfire\spatiotemporal_wildfire_gnn\reports\checkpoints\phase5a_gcn_best.pt

Next run:
python scripts/phase5a_make_figures.py


Sage:

 scripts/phase5a_run_gnn.py --model graphsage --epochs 50 --device cpu

===========================================================================
Phase 5A — Deterministic GNN Architecture
===========================================================================
Loading graph: D:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Nodes     : 327,405
Features  : 61
Edges     : 2,511,084
Train     : 237,304
Val       : 32,570
Test      : 57,531

Model architecture:
GraphSAGEModel(
  (convs): ModuleList(
    (0): SAGEConv(61, 64, aggr=mean)
    (1-2): 2 x SAGEConv(64, 64, aggr=mean)
  )
  (head): MLPHead(
    (net): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=64, out_features=1, bias=True)
    )
  )
)

Training started...
Epoch 001/50 | train_loss=482.09665 | val_loss=1105.94177 | lr=1.00e-03
Epoch 005/50 | train_loss=85.65029 | val_loss=7.82571 | lr=1.00e-03
Epoch 010/50 | train_loss=11.73873 | val_loss=0.72938 | lr=1.00e-03
Epoch 015/50 | train_loss=2.42194 | val_loss=1.44303 | lr=1.00e-03
Epoch 020/50 | train_loss=1.15096 | val_loss=1.49840 | lr=5.00e-04
Early stopping at epoch 22

Training finished in 83.6s
Best val_loss: 0.72938

Predicting on test split...

  ── GRAPHSAGE ──
    R²       =   0.4424   (>0 = beats naive mean predictor)
    MAE      =  0.01688  (burn prob scale ~[0, 0.25])
    Spearman =   0.7468   (rank correlation, robust)
    Brier    =  0.00078  (lower = better calibration)
    ECE      =  0.01066  (target < 0.05 after calibration)
    n_test   =   57,531

    Binned by BP quantile:
    Bin    BP range                      n       R²        MAE   Spearman
    --------------------------------------------------------------------
    1      [0.0000, 0.0040]    11,506  -81.527    0.00943     -0.094
    2      [0.0040, 0.0099]    11,505  -15.829    0.00610      0.191
    3      [0.0099, 0.0205]    11,507   -6.176    0.00440      0.154
    4      [0.0205, 0.0465]    11,506   -6.587    0.01409      0.314
    5      [0.0465, 0.2082]    11,507   -1.379    0.05037      0.625 ← HIGH RISK

===========================================================================
Phase 5A complete
===========================================================================
Saved metrics     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_graphsage_metrics.csv
Saved binned      : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_graphsage_binned_metrics.csv
Saved history     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_graphsage_history.csv
Saved predictions : D:\wildfire\spatiotemporal_wildfire_gnn\reports\predictions\phase5a_graphsage_test_predictions.npz       
Saved checkpoint  : D:\wildfire\spatiotemporal_wildfire_gnn\reports\checkpoints\phase5a_graphsage_best.pt

Next run:
python scripts/phase5a_make_figures.py


GAT:

_gnn>python scripts/phase5a_run_gnn.py --model gat --epochs 50 --device cpu --hidden-channels 16 --heads 2 --num-layers 2 --dropout 0.3

===========================================================================
Phase 5A — Deterministic GNN Architecture
===========================================================================
Loading graph: D:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Nodes     : 327,405
Features  : 61
Edges     : 2,511,084
Train     : 237,304
Val       : 32,570
Test      : 57,531

Model architecture:
GATModel(
  (convs): ModuleList(
    (0): GATConv(61, 16, heads=2)
    (1): GATConv(32, 16, heads=2)
  )
  (head): MLPHead(
    (net): Sequential(
      (0): Linear(in_features=32, out_features=16, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=16, out_features=1, bias=True)
    )
  )
)

Training started...
Epoch 001/50 | train_loss=9362.38965 | val_loss=2052.49585 | lr=1.00e-03
Epoch 005/50 | train_loss=5050.90967 | val_loss=56.62851 | lr=1.00e-03
Epoch 010/50 | train_loss=2637.91528 | val_loss=576.14026 | lr=1.00e-03
Epoch 015/50 | train_loss=1583.20667 | val_loss=985.64069 | lr=5.00e-04
Early stopping at epoch 17

Training finished in 69.4s
Best val_loss: 56.62851

Predicting on test split...

  ── GAT ──
    R²       = -15.4094   (>0 = beats naive mean predictor)
    MAE      =  0.12658  (burn prob scale ~[0, 0.25])
    Spearman =   0.3540   (rank correlation, robust)
    Brier    =  0.02290  (lower = better calibration)
    ECE      =  0.12504  (target < 0.05 after calibration)
    n_test   =   57,531

    Binned by BP quantile:
    Bin    BP range                      n       R²        MAE   Spearman
    --------------------------------------------------------------------
    1      [0.0000, 0.0040]    11,506 -13925.659    0.09539      0.219
    2      [0.0040, 0.0099]    11,505 -7265.624    0.11686      0.083
    3      [0.0099, 0.0205]    11,507 -3214.022    0.14808      0.148
    4      [0.0205, 0.0465]    11,506 -579.860    0.15361     -0.001
    5      [0.0465, 0.2082]    11,507  -11.052    0.11898      0.226 ← HIGH RISK

===========================================================================
Phase 5A complete
===========================================================================
Saved metrics     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gat_metrics.csv
Saved binned      : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gat_binned_metrics.csv
Saved history     : D:\wildfire\spatiotemporal_wildfire_gnn\reports\tables\phase5a_gat_history.csv
Saved predictions : D:\wildfire\spatiotemporal_wildfire_gnn\reports\predictions\phase5a_gat_test_predictions.npz
Saved checkpoint  : D:\wildfire\spatiotemporal_wildfire_gnn\reports\checkpoints\phase5a_gat_best.pt

Next run:
python scripts/phase5a_make_figures.py


# 04 — GNN Experiments · Phase 5A

**Purpose:** Load trained GNN, verify all assertions, analyse predictions, compare against baselines.

**Run AFTER:** `python scripts/phase5a_train_gnn.py --arch GAT`

**Target:** R² > 0.7187 (CNN baseline, the strongest baseline from Phase 4B)

**Critical rule:** NEVER re-apply QuantileTransformer at eval — `y` is already transformed. Only `inverse_transform` before metrics.
---

In [ ]:
import os, sys, json, pickle
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_gnn.utils import load_yaml_config, set_seed
from wildfire_gnn.models.gnn import build_model, count_parameters
from wildfire_gnn.models.gnn_pipeline import GNNPipeline
from wildfire_gnn.evaluation.metrics import (
    r2_score, mae_score, spearman_rho, brier_score,
    expected_calibration_error, binned_metrics
)

config = load_yaml_config(PROJECT_ROOT / 'configs' / 'gnn_config.yaml')
set_seed(config['training']['seed'])

p            = config['paths']
GRAPH_PATH   = PROJECT_ROOT / p['graph_data']
TRANS_PATH   = PROJECT_ROOT / p['target_transformer']
CKPT_DIR     = PROJECT_ROOT / 'checkpoints'
TBL_DIR      = PROJECT_ROOT / 'reports' / 'tables'
FIG_DIR      = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load baselines for comparison
BASELINES = {}
for csv in ['phase4_baseline_metrics.csv', 'phase4b_cnn_metrics.csv']:
    path = TBL_DIR / csv
    if path.exists():
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            BASELINES[row['model']] = row.to_dict()

print(f'Project root  : {PROJECT_ROOT}')
print(f'Baselines loaded: {list(BASELINES.keys())}')

In [ ]:
graph = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)

print('Graph loaded:')
print(f'  num_nodes     : {graph.num_nodes:,}')
print(f'  num_features  : {graph.num_node_features}')
print(f'  train         : {int(graph.train_mask.sum()):,}')
print(f'  val           : {int(graph.val_mask.sum()):,}')
print(f'  test          : {int(graph.test_mask.sum()):,}')
print(f'  y mean        : {float(graph.y.mean()):.4f}  (should be near 0)')
print(f'  y std         : {float(graph.y.std()):.4f}   (should be near 1)')

# Assertions
assert graph.num_node_features == 61, f'Expected 61, got {graph.num_node_features}'
assert abs(float(graph.y.mean())) < 0.5, 'y not transformed or double-transformed!'
assert (graph.train_mask & graph.val_mask).sum() == 0, 'Train/Val overlap!'
assert (graph.train_mask & graph.test_mask).sum() == 0, 'Train/Test overlap!'
assert graph.val_mask.sum() > 0, 'val_mask is zero!'
print('\n✓ All graph assertions passed')

In [ ]:
ARCH = 'GAT'   # Change to 'GCN' or 'GraphSAGE' for ablation
ckpt_path = CKPT_DIR / f'gnn_{ARCH.lower()}_best.pt'

if not ckpt_path.exists():
    print(f'Checkpoint not found: {ckpt_path}')
    print(f'Run: python scripts/phase5a_train_gnn.py --arch {ARCH}')
else:
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model = build_model(
        architecture = config['model']['architecture'],
        in_channels  = config['model']['in_channels'],
        hidden       = config['model']['hidden_channels'],
        num_layers   = config['model'].get('num_layers', 4),
        heads        = config['model'].get('heads', 8),
        dropout      = config['model'].get('dropout', 0.3),
    )
    model.load_state_dict(ckpt['model_state'])
    print(f'Model loaded: {ARCH}')
    print(f'Parameters : {count_parameters(model):,}')
    print(f'Architecture: {model}')

In [ ]:
history_path = TBL_DIR / f'phase5a_{ARCH.lower()}_history.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    print(history.tail())

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(history['epoch'], history['train_loss'], label='Train', lw=2)
    ax.plot(history['epoch'], history['val_loss'],   label='Val',   lw=2)
    best_epoch = history.loc[history['val_loss'].idxmin(), 'epoch']
    best_val   = history['val_loss'].min()
    ax.axvline(best_epoch, color='red', ls='--', alpha=0.6,
               label=f'Best val (epoch {best_epoch:.0f}, loss={best_val:.4f})')

    # Warning zone for generalization gap
    final_train = history['train_loss'].iloc[-1]
    final_val   = history['val_loss'].iloc[-1]
    gap = final_val - final_train
    ax.set_title(f'Phase 5A {ARCH} Training Curve\n'
                 f'train={final_train:.4f}  val={final_val:.4f}  '
                 f'gap={gap:.4f}  '
                 f'({"⚠ Overfitting" if gap > 0.3 else "✓ OK"})')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (Gaussian NLL)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'p5a_{ARCH.lower()}_loss.png', dpi=150)
    plt.show()
    print('Figure saved')

In [ ]:
N_MC = 30
print(f'Running {N_MC} MC Dropout forward passes...')

# Keep dropout ON for MC Dropout
model.train()
sample_means = []
sample_logvars = []

with torch.no_grad():
    for i in range(N_MC):
        mean, lv = model(graph.x, graph.edge_index)
        sample_means.append(mean[graph.test_mask].numpy())
        sample_logvars.append(lv[graph.test_mask].numpy())

samples   = np.stack(sample_means)     # (N_MC, N_test)
mean_pred = samples.mean(axis=0)       # epistemic: mean
std_pred  = samples.std(axis=0)        # epistemic: uncertainty
aleatoric = np.sqrt(np.exp(np.stack(sample_logvars).mean(axis=0)))  # aleatoric
total_unc = np.sqrt(aleatoric**2 + std_pred**2)

print(f'MC predictions computed:')
print(f'  mean_pred : min={mean_pred.min():.3f}  max={mean_pred.max():.3f}')
print(f'  epistemic : mean={std_pred.mean():.4f}  max={std_pred.max():.4f}')
print(f'  aleatoric : mean={aleatoric.mean():.4f}  max={aleatoric.max():.4f}')
print(f'  total_unc : mean={total_unc.mean():.4f}')

In [ ]:
# CRITICAL: inverse transform BEFORE any metric
with open(TRANS_PATH, 'rb') as f:
    transformer = pickle.load(f)

y_pred_bp = transformer.inverse_transform(mean_pred.reshape(-1,1)).ravel()
y_true_bp = graph.y_raw[graph.test_mask].numpy().ravel()

r2  = r2_score(y_true_bp, y_pred_bp)
mae = mae_score(y_true_bp, y_pred_bp)
spr = spearman_rho(y_true_bp, y_pred_bp)
bri = brier_score(y_true_bp, y_pred_bp)
ece = expected_calibration_error(y_true_bp, y_pred_bp)

print(f'\n  ── {ARCH} Results (test split, original BP scale) ──')
print(f'  R²       = {r2:.4f}')
print(f'  MAE      = {mae:.5f}')
print(f'  Spearman = {spr:.4f}')
print(f'  Brier    = {bri:.5f}')
print(f'  ECE      = {ece:.5f}')
print(f'  n_test   = {len(y_true_bp):,}')

print(f'\n  Comparison vs baselines:')
for name, row in BASELINES.items():
    diff = r2 - row.get("r2", 0)
    sym  = "✓" if diff > 0 else "✗"
    print(f'  {sym} {name:<22} R²={row.get("r2",0):.4f}  diff={diff:+.4f}')

In [ ]:
rng = np.random.default_rng(42)
idx = rng.choice(len(y_true_bp), min(20_000, len(y_true_bp)), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: prediction vs truth
ax = axes[0]
ax.scatter(y_true_bp[idx], y_pred_bp[idx], s=2, alpha=0.3)
lo = min(y_true_bp.min(), y_pred_bp.min())
hi = max(y_true_bp.max(), y_pred_bp.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5)
ax.set_xlabel('True Burn Probability')
ax.set_ylabel('Predicted Burn Probability')
ax.set_title(f'{ARCH} — Predicted vs True\nR²={r2:.4f}  MAE={mae:.4f}')

# Right: uncertainty map (uncertainty vs true BP)
ax2 = axes[1]
ax2.scatter(y_true_bp[idx], total_unc[idx], s=2, alpha=0.3, color='orange')
ax2.set_xlabel('True Burn Probability')
ax2.set_ylabel('Total Uncertainty (epistemic + aleatoric)')
ax2.set_title(f'{ARCH} — Uncertainty vs True BP\n'
              f'High uncertainty on high-risk cells = Gap 1+2 addressed')

plt.tight_layout()
plt.savefig(FIG_DIR / f'p5a_{ARCH.lower()}_scatter.png', dpi=150)
plt.show()
print('Figure saved')

In [ ]:
bins_gnn = binned_metrics(y_true_bp, y_pred_bp)

print(f'  Binned evaluation ({ARCH}) vs XGBoost and CNN:')
print(f'  {"Bin":<6} {"BP range":<22} {"n":>8} {"GNN R²":>8} '
      f'{"GNN MAE":>10} {"GNN Spearman":>13}')
print(f'  {"-"*70}')

for b in bins_gnn:
    hi_flag = " ← HIGH RISK" if b["bin"] == len(bins_gnn) else ""
    print(f'  {b["bin"]:<6} [{b["bin_low"]:.4f}, {b["bin_high"]:.4f}]  '
          f'{b["n"]:>8,} {b["r2"]:>8.3f} {b["mae"]:>10.5f} '
          f'{b["spearman"]:>13.3f}{hi_flag}')

# Check if GNN beats CNN in high-risk bin
bin5_gnn = [b for b in bins_gnn if b['bin'] == 5]
if bin5_gnn:
    print(f'\n  High-risk bin (bin 5):')
    print(f'    CNN MAE  = 0.02789  (Phase 4B result)')
    print(f'    GNN MAE  = {bin5_gnn[0]["mae"]:.5f}')
    if bin5_gnn[0]['mae'] < 0.02789:
        print(f'    ✓ GNN beats CNN on high-risk cells!')
    else:
        print(f'    ✗ GNN does not beat CNN on high-risk cells yet')

In [ ]:
all_results = list(BASELINES.values()) + [{
    'model': ARCH, 'r2': r2, 'mae': mae,
    'spearman': spr, 'brier': bri, 'ece': ece
}]

df_all = pd.DataFrame(all_results)[['model','r2','mae','spearman','brier','ece']]
df_all = df_all.sort_values('r2', ascending=False).reset_index(drop=True)

print('\n  FULL COMPARISON TABLE')
print('  (test split, original BP scale, geographic split)')
print(df_all.to_string(index=False))

# Save combined table
combined_path = TBL_DIR / 'phase5a_all_models_comparison.csv'
df_all.to_csv(combined_path, index=False)
print(f'\n  Saved: {combined_path.name}')

In [ ]:
print('='*55)
print('  PHASE 5A COMPLETION CHECKLIST')
print('='*55)

items = [
    ('GAT model trained',      (CKPT_DIR / 'gnn_gat_best.pt').exists()),
    ('GCN model trained',      (CKPT_DIR / 'gnn_gcn_best.pt').exists()),
    ('GraphSAGE trained',      (CKPT_DIR / 'gnn_graphsage_best.pt').exists()),
    ('R² > 0.7187 (CNN)',      r2 > 0.7187),
    ('R² > 0.6761 (XGBoost)', r2 > 0.6761),
    ('val_mask non-zero',      int(graph.val_mask.sum()) > 0),
    ('No geographic leakage',  int((graph.train_mask & graph.test_mask).sum()) == 0),
    ('MC Dropout ran (30 passes)', len(sample_means) == N_MC),
    ('Predictions saved',      (TBL_DIR / f'phase5a_{ARCH.lower()}_metrics.csv').exists()),
]

all_ok = True
for label, ok in items:
    sym = '✓' if ok else '✗'
    print(f'  {sym}  {label}')
    all_ok = all_ok and ok

print('='*55)
if all_ok:
    print('  ALL CHECKS PASSED — proceed to Phase 5B (Calibration)')
else:
    print('  SOME CHECKS FAILED')
    if r2 <= 0.7187:
        print('  Hint: GNN does not beat CNN yet.')
        print('  Try: increase hidden_channels to 512, decrease dropout to 0.2')
        print('  Or: add edge_attr (distance), use graph.pos as positional encoding')